# 02 — FDTD physics check

Short derivation **and** validation of the solver, rendered with the project's pygame renderer (no matplotlib). The full derivation and error analysis are in `report/Final_Report.docx`.

In [ ]:
import sys, pathlib, os
SRC = pathlib.Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")   # headless: render to images, no window

import numpy as np
import pygame
from IPython.display import Image, display

from config import MATERIALS
from grid import RoomGrid
from utils import coord_to_cell, cell_to_coord, empty_field, pressure_reflection
from render import field_to_rgb, to_surface
import scenes

pygame.init()

def show(alpha, field=None, p_scale=1.0, scale=5, fname="_render.png"):
    """Render an (NY, NX) alpha-map (+ optional pressure field) with the project's
    own pygame renderer and display it inline -- no matplotlib."""
    if field is None:
        field = np.zeros_like(alpha)
    surf = to_surface(field_to_rgb(field, alpha, p_scale))
    surf = pygame.transform.scale(surf, (alpha.shape[1] * scale, alpha.shape[0] * scale))
    pygame.image.save(surf, fname)
    display(Image(filename=fname))

from room import Room
from physics_solver import WaveSolver
from render import draw_line_chart

def show_field(solver, alpha, scale=5, fname="_field.png"):
    peak = max(float(np.max(np.abs(solver.field))), 1e-4)   # per-frame contrast
    show(alpha, solver.field, p_scale=peak, scale=scale, fname=fname)

def run_to(solver, step):
    while solver.n < step:
        solver.step()

## The scheme

Damped wave equation `u_tt + beta*u_t = c**2 * laplacian(u)`, discretised with a 3-level time scheme and the 5-point Laplacian. With `r = c*dt/dx`:

    u_next = A*u_curr - B*u_prev + C*L
    A=(2-beta*dt)/d, B=(1-beta*dt/2)/d, C=r**2/d, d=1+beta*dt/2

`beta=0` is the lossless leapfrog scheme; CFL stability needs `r <= 1/sqrt(2)`. Walls (α>0) reflect via a ghost `g = u - (1-R_p)/r*(u - u_prev)`, `R_p = sqrt(1-alpha)`. The clap is a zero-mean Ricker wavelet.

## A clap propagating

Compression shows **red** on a white background; the wavefront expands, reflects, and diffracts through the doorway.

In [ ]:
grid = RoomGrid()
room = scenes.two_rooms(grid)
solver = WaveSolver(grid, room.alpha)
solver.add_impulse(2.0, 3.0)
for step in (90, 180, 320):
    run_to(solver, step)
    print(f"step {step}  t = {solver.t*1e3:.1f} ms")
    show_field(solver, room.alpha)

## Energy: rigid conserves vs absorbing decays

With `beta=0`, a rigid room conserves energy; absorbing walls make it decay to zero.

In [ ]:
def etrace(alpha_map, n=1000):
    s = WaveSolver(grid, alpha_map); s.add_impulse(4.0, 3.0)
    return np.array(s.run(n, record_energy=True))

Er = etrace(Room(grid).alpha)
Ef = etrace(Room(grid).add_border("Acoustic Foam").alpha)

W, H = 720, 280
surf = pygame.Surface((W, H)); surf.fill((255, 255, 255))
plot = (50, 30, W - 80, H - 70)
def line(E, col):
    m = E.max() or 1.0
    pts = [(plot[0] + (plot[2]-1)*i/(len(E)-1), plot[1] + (plot[3]-1)*(1 - min(v/m, 1.0)))
           for i, v in enumerate(E)]
    pygame.draw.lines(surf, col, False, pts, 2)
pygame.draw.rect(surf, (208, 208, 208), plot, 1)
line(Er, (28, 28, 28)); line(Ef, (150, 150, 150))
f = pygame.font.SysFont("arial", 15)
surf.blit(f.render("rigid (conserves)", True, (28, 28, 28)), (60, 36))
surf.blit(f.render("Acoustic Foam (decays)", True, (150, 150, 150)), (60, 54))
pygame.image.save(surf, "_energy.png"); display(Image(filename="_energy.png"))

## Verdict

All quantitative checks (stability, energy drift, wave speed, absorption ladder) are automated in `src/verify.py` — run `python verify.py` for the PASS/FAIL report.